### RERANKING 

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
loader = TextLoader("langchain_sampple.txt")
raw_text = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap = 50)
docs = splitter.split_documents(raw_text)

docs



In [10]:
query = "How does reranking improve a RAG system?"

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


embedding = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)
vectorestore = FAISS.from_documents(docs,embedding)

retriever = vectorestore.as_retriever(search_kwargs = {"k":8})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model = "gemini-flash-latest",
    temperature = 0
)
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.2'}}, output_version=None, profile={'name': 'Gemini Flash Latest', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-flash-latest', temperature=0.0, client=<google.genai.client.Client object at 0x00000221B4488CD0>, default_metadata=(), model_kwargs={})

In [25]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000022149B6A510>, search_kwargs={'k': 8})

In [33]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ-API-KEY"] = os.getenv("GROQ_API_KEY")

llm =init_chat_model("groq:openai/gpt-oss-120b")
llm



ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000221B79DDE50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000221B79DE350>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [34]:
prompt = PromptTemplate.from_template(
    """
    Given the query and documents, rank the documents
    according to their relevance.

    Query:
    {query}

    Documents:
    {documents}

    Instructions:
    - Think about the relevance of each document to the user Query.
    - return a list of document indices in ranked order , strting from the most relevant....
    
    """
)

In [35]:
retrieved_docs = retriever.invoke(query)
retrieved_docs

[Document(id='5699a4ea-2385-4250-9626-fc2b6b6c4e7f', metadata={'source': 'langchain_sampple.txt'}, page_content='A reranker receives the query and candidate documents and calculates a more accurate relevance score.\nThe highest scoring documents are selected as the final context for the language model.\nReranking is usually performed after an initial retrieval stage because rerankers are more computationally expensive.\nA common RAG pipeline can therefore be described as query, retrieval, reranking, and generation.\nFor example, a user may ask how to implement authentication in a FastAPI application.'),
 Document(id='ce8734d3-fe05-40c1-b81c-c663569eea0a', metadata={'source': 'langchain_sampple.txt'}, page_content='Python is widely used in AI, Machine Learning, backend development, automation, and data analysis.\nA good RAG system should retrieve relevant information, minimize irrelevant context, and provide accurate answers.\nEvaluation is important for measuring retrieval quality, ans

In [36]:
chain = prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'query'], input_types={}, partial_variables={}, template='\n    Given the query and documents, rank the documents\n    according to their relevance.\n\n    Query:\n    {query}\n\n    Documents:\n    {documents}\n\n    Instructions:\n    - Think about the relevance of each document to the user Query.\n    - return a list of document indices in ranked order , strting from the most relevant....\n\n    ')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachmen

In [37]:
doc_line = [f"{i+1}.{doc.page_content}" for i ,doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_line)

In [47]:
response = chain.invoke({
    "query": query,
    "documents": formatted_docs
})



In [51]:
indices = [
    int(x.strip()) - 1
    for x in response
    if x.strip().isdigit()
]

print(indices)

[0, 1, 5, 3, 2, 6, 7, 4]


In [53]:
ranked_docs = [retrieved_docs[i] for i in indices if 0<= i < len(retrieved_docs)]

In [54]:


for rank, doc in enumerate(ranked_docs, start=1):
    print(f"\nRank {rank}")
    print(doc.page_content)


Rank 1
A reranker receives the query and candidate documents and calculates a more accurate relevance score.
The highest scoring documents are selected as the final context for the language model.
Reranking is usually performed after an initial retrieval stage because rerankers are more computationally expensive.
A common RAG pipeline can therefore be described as query, retrieval, reranking, and generation.
For example, a user may ask how to implement authentication in a FastAPI application.

Rank 2
Python is widely used in AI, Machine Learning, backend development, automation, and data analysis.
A good RAG system should retrieve relevant information, minimize irrelevant context, and provide accurate answers.
Evaluation is important for measuring retrieval quality, answer quality, latency, and overall system performance.
Reranking can improve the precision of retrieved context and is especially useful when the initial retriever returns many candidate documents.

Rank 3
The retriever 